# pycheckers rules and board validation

Visual checks for board-state representation and American checkers rules. This notebook intentionally avoids tablebase generation.

In [ ]:
%matplotlib inline

from pycheckers import (
    BoardState,
    legal_successors,
    moves_df,
    show_state,
    show_turn,
    square_mask,
    turns_df,
)
from pycheckers.bitboard import generate_move_templates

## Move Template Sanity Check

The rules engine starts from primitive diagonal move templates over playable dark squares.

In [ ]:
templates = generate_move_templates()
{
    "templates": len(templates),
    "quiet_templates": sum(not move["is_capture"] for move in templates),
    "capture_templates": sum(move["is_capture"] for move in templates),
}

## Initial Board

The package-level state object is immutable and hashable. It stores black pieces, white pieces, kings, and side to move.

In [ ]:
state = BoardState.initial()
{
    "black_count": state.black_count,
    "white_count": state.white_count,
    "king_count": state.king_count,
    "side": state.side,
}

In [ ]:
show_state(state, size=3);

## Opening Legal Moves

From the initial position there are seven legal quiet moves for black.

In [ ]:
opening_moves = state.legal_moves()
assert len(opening_moves) == 7
assert not any(move["is_capture"] for move in opening_moves)
moves_df(state)

In [ ]:
destination_dots = 0
for move in opening_moves:
    destination_dots |= move["to_mask"]
show_state(state, dots=destination_dots, size=3, title="Initial legal destinations");

## Applying A Move

A primitive move produces a new `BoardState`; after a non-capture move the side to move switches.

In [ ]:
opening_move = opening_moves[0]
next_state = state.apply_move(opening_move)
{
    "before_side": state.side,
    "after_side": next_state.side,
    "before_black_count": state.black_count,
    "after_black_count": next_state.black_count,
}

In [ ]:
state.show(other=next_state.as_tuple()[:3], size=2);

## Forced Capture

If any capture is legal, non-capturing moves are not returned.

In [ ]:
capture_state = BoardState(
    square_mask(2, 1),
    square_mask(3, 2),
    0,
    "B",
)
capture_moves = capture_state.legal_moves()
assert len(capture_moves) == 1
assert all(move["is_capture"] for move in capture_moves)
show_state(capture_state, size=3, title="Forced capture");
moves_df(capture_state)

## Multi-Jump Turn

A legal turn can contain more than one primitive capture. The side does not switch until the full turn finishes.

In [ ]:
jump_state = BoardState(
    square_mask(2, 1),
    square_mask(3, 2) | square_mask(5, 4),
    0,
    "B",
)
jump_turns = jump_state.legal_turns()
assert len(jump_turns) == 1
assert len(jump_turns[0]) == 2
turns_df(jump_state, jump_turns)

In [ ]:
show_turn(jump_state, jump_turns[0], size=2);
[s.side for s in jump_state.turn_states(jump_turns[0])]

## Promotion

A man that reaches the crown row becomes a king. If it crowns during a jump, the capture chain stops there under American checkers rules.

In [ ]:
promotion_state = BoardState(
    square_mask(6, 1),
    square_mask(0, 1),
    0,
    "B",
)
promotion_moves = promotion_state.legal_moves()
assert len(promotion_moves) == 2
assert moves_df(promotion_state)["promotes"].all()
show_state(promotion_state, size=3, title="Promotion choices");
moves_df(promotion_state)

In [ ]:
promotion_successor = legal_successors(promotion_state)[0]
show_turn(promotion_state, promotion_state.legal_turns()[0], size=2);
{
    "successor_side": promotion_successor.side,
    "successor_kings": promotion_successor.king_count,
}

## Kings Move Backward

Men only move forward. Kings can move and capture in both diagonal directions.

In [ ]:
man_state = BoardState(square_mask(3, 2), 0, 0, "B")
king_state = BoardState(square_mask(3, 2), 0, square_mask(3, 2), "B")
{
    "man_destinations": sorted(zip(moves_df(man_state)["to_r"], moves_df(man_state)["to_c"])),
    "king_destinations": sorted(zip(moves_df(king_state)["to_r"], moves_df(king_state)["to_c"])),
}

In [ ]:
show_state(king_state, size=3, title="Black king can move backward");
moves_df(king_state)

## Invalid Board States

Construction validates overlap, dark-square occupancy, king subset, and side-to-move.

In [ ]:
invalid_cases = {
    "off_dark_square": lambda: BoardState(square_mask(0, 0), 0, 0, "B"),
    "overlap": lambda: BoardState(square_mask(2, 1), square_mask(2, 1), 0, "B"),
    "king_without_piece": lambda: BoardState(square_mask(2, 1), 0, square_mask(3, 2), "B"),
    "bad_side": lambda: BoardState(square_mask(2, 1), 0, 0, "red"),
}
errors = {}
for name, factory in invalid_cases.items():
    try:
        factory()
    except ValueError as exc:
        errors[name] = str(exc)
errors